# RAG Pipeline with Embeddings: Practice Exercise

Build a Retrieval-Augmented Generation system to answer technical support questions using TechMart's troubleshooting knowledge base. You will implement the complete RAG workflow using the patterns from the lesson.

**What you'll implement:**
- The RAG state class to track query, context, and answer
- The retrieve node for similarity search
- The generate node for creating grounded responses
- The LangGraph workflow connecting all components

**Estimated time:** 15-20 minutes

In [ ]:
!pip install langchain-core langchain-openai langgraph python-dotenv langchain-chroma langchain-text-splitters

In [ ]:
import os
from functools import lru_cache

DEFAULT_REQUIRED_KEYS = ("OPENAI_API_KEY",)

@lru_cache(maxsize=1)
def configure_environment(required_keys=None):
    """
    Factory function to configure environment variables.
    Executes once and caches results.
    """
    if required_keys is None:
        required_keys = ("OPENAI_API_KEY", )

    IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ

    if IN_COLAB:
        from google.colab import userdata
        print("Configuring for Google Colab environment...")
        for key in required_keys:
            try:
                os.environ[key] = userdata.get(key)
            except Exception:
                print(f"Warning: Could not find {key} in Colab secrets.")
    else:
        from dotenv import load_dotenv
        print("Configuring for local environment...")
        load_dotenv()

    # Validation
    for key in required_keys:
        if not os.getenv(key):
            raise ValueError(f"Missing required environment variable: {key}")

    return True

## Setup

Run this cell to import all required libraries and initialize the API client.

In [ ]:
# Setup - run this cell first

import os
import csv

from typing import TypedDict, List

# LangChain imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

# LangGraph imports
from langgraph.graph import StateGraph, START, END

# Load environment variables
configure_environment()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please set it in your .env file.")

print("Setup complete!")

Configuring for Google Colab environment...
Setup complete!


## Context

You are building a technical support assistant for TechMart that can answer customer troubleshooting questions. The knowledge base contains troubleshooting guides for various tech products including laptops, monitors, mice, and desktop computers.

**Your task:**
1. Define the RAG state class to track the workflow data
2. Initialize the LLM and create an appropriate prompt template
3. Implement the retrieve node that finds relevant troubleshooting guides
4. Implement the generate node that creates helpful answers
5. Build the LangGraph workflow connecting these components

**Input:** A customer question about a technical issue (string)

**Output:** A grounded troubleshooting response based on the knowledge base

## Knowledge Base Setup (Provided)

The following cells load the TechMart troubleshooting data and create the vector store. Run them to prepare the knowledge base.

In [ ]:
import csv
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Path to the file in Google Drive
data_path = '/content/drive/MyDrive/Colab Notebooks/Agentic-AI/Level_3/techmart_troubleshooting.csv'

if not os.path.exists(data_path):
    print(f"Error: File not found at {data_path}. Please ensure it is uploaded to your Drive.")
    troubleshooting_guides = []
else:
    troubleshooting_guides = []
    with open(data_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            troubleshooting_guides.append(row)

    if troubleshooting_guides:
        print(f"Loaded {len(troubleshooting_guides)} troubleshooting guides")
        print(f"\nColumns: {list(troubleshooting_guides[0].keys())}")
        print(f"\nSample guide:")
        sample = troubleshooting_guides[0]
        print(f"  Issue: {sample['issue_title']}")
        print(f"  Product: {sample['product_id']}")
        print(f"  Symptoms: {sample['symptoms'][:100]}...")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 154 troubleshooting guides

Columns: ['issue_title', 'product_id', 'symptoms', 'steps']

Sample guide:
  Issue: Wireless mouse disconnects randomly
  Product: SKU010
  Symptoms: Cursor freezes for 1–2 seconds, USB‑dongle LED blinks red, occurs every 5–10 minutes....


In [ ]:
# Create documents and build vector store - run this cell

# Convert to LangChain Documents
# Each document combines issue, symptoms, and troubleshooting steps
support_docs = []
for guide in troubleshooting_guides:
    content = f"""Issue: {guide['issue_title']}
Symptoms: {guide['symptoms']}
Troubleshooting Steps:
{guide['steps']}"""

    doc = Document(
        page_content=content,
        metadata={
            "issue_title": guide["issue_title"],
            "product_id": guide["product_id"]
        }
    )
    support_docs.append(doc)

# Split into chunks for better retrieval
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)
support_chunks = splitter.split_documents(support_docs)

# Create embeddings and vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

support_vectorstore = Chroma(
    collection_name="techmart_support",
    embedding_function=embeddings
)
support_vectorstore.add_documents(documents=support_chunks)

print(f"Created {len(support_chunks)} chunks from {len(support_docs)} guides")
print("Vector store ready!")

Created 305 chunks from 154 guides
Vector store ready!


In [ ]:
# Test the vector store - run this cell to verify it works

test_query = "My laptop battery drains too fast"
test_results = support_vectorstore.similarity_search(test_query, k=2)

print(f"Test query: '{test_query}'")
print(f"\nFound {len(test_results)} results:")
for i, doc in enumerate(test_results, 1):
    print(f"\n[{i}] {doc.metadata.get('issue_title', 'Unknown')}")
    print(f"    {doc.page_content[:150]}...")

Test query: 'My laptop battery drains too fast'

Found 2 results:

[1] UltraBook Pro battery drains quickly
    Issue: UltraBook Pro battery drains quickly
Symptoms: Battery indicator drops from 100 % to 70 % in under an hour of web browsing.
Troubleshooting Ste...

[2] Laptop Battery Not Charging When Plugged In
    4. Disconnect the charger, power off the laptop, and hold the power button for 15 seconds. Then reconnect the charger and power on the device. 
5. In ...


## Part 1: Define the RAG State

Define the state class that will be passed between nodes in your workflow.

In [ ]:
class SupportRAGState(TypedDict):
    """
    State for the TechMart support RAG workflow.

    This TypedDict should contain three fields:
    - query: The customer's support question (str)
    - context: Retrieved troubleshooting documents (List[Document])
    - answer: The generated support response (str)
    """
    # TODO: Define the three state fields with appropriate types
    # Hint: Use str for text fields and List[Document] for retrieved docs
    query: str
    context: List[Document]
    answer: str

## Part 2: Initialize LLM and Prompt

Create the language model instance and define a prompt template for technical support responses.

In [ ]:
# TODO 1: Initialize the ChatOpenAI model
# Use model="gpt-4o-mini" and temperature=0 for consistent responses
support_llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)  # Replace with ChatOpenAI initialization

# TODO 2: Create a ChatPromptTemplate for technical support
# The template should:
# - Have a system message that instructs the LLM to act as a tech support agent
# - Tell it to use ONLY the provided context to answer
# - Include a {context} placeholder for retrieved documents
# - Have a human message with a {query} placeholder
#
# Use ChatPromptTemplate.from_messages() with a list of tuples:
# [("system", "..."), ("human", "{query}")]

support_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """You are a support assistant for TechMart, an electronics retailer.
        your job is to help customer trouble shooting their tech products
       Use Technical support guides, problem diagnosis, step-by-step solutions, and fixes for product issues that TechMart sells.
       if the guides don't contain relevant information, inform the user to contact support.
      Troubleshooting Guides:
     {context}"""
     ),
     (
        "human", "{query}"
     )
]
)

print("LLM and prompt initialized")

LLM and prompt initialized


## Part 3: Implement the Nodes

Implement the retrieve and generate nodes for the RAG workflow.

### Retrieve Node

This node takes the customer query and retrieves relevant troubleshooting documents.

In [ ]:
def retrieve_support_docs(state: SupportRAGState) -> SupportRAGState:
    """
    Retrieve relevant troubleshooting documents based on customer query.

    This function should:
    1. Extract the query from the state
    2. Use support_vectorstore.similarity_search() to find 4 relevant documents
    3. Return a new state dict with query, context (retrieved docs), and empty answer

    Args:
        state: Current workflow state containing the customer query

    Returns:
        Updated state dict with 'context' populated with retrieved Document objects
    """
    # TODO: Implement retrieval logic
    # 1. Get query from state
    # 2. Call support_vectorstore.similarity_search(query, k=4)
    # 3. Return dict with query, context, and empty answer
    query = state["query"]
    retrieved_docs = support_vectorstore.similarity_search(query, k=4)
    return {
        "query": query,
        "context": retrieved_docs,
        "answer": ""
    }

### Generate Node

This node uses the retrieved context to generate a helpful support response.

In [ ]:
def generate_support_response(state: SupportRAGState) -> SupportRAGState:
    """
    Generate a support response using retrieved troubleshooting context.

    This function should:
    1. Extract query and context documents from state
    2. Format context docs into a single string (include issue title and content)
    3. Use support_prompt.invoke() with context and query
    4. Call support_llm.invoke() with the formatted prompt
    5. Return updated state with the answer

    Args:
        state: Current workflow state with query and retrieved context

    Returns:
        Updated state dict with 'answer' populated
    """
    # TODO: Implement generation logic
    # 1. Get query and context from state
    # 2. Format context: join docs with titles, e.g.:
    #    "[Issue: {title}]\n{content}" for each doc
    # 3. Invoke prompt with {"context": context_str, "query": query}
    # 4. Invoke LLM with the prompt result
    # 5. Return dict with query, context, and answer (response.content)
    query = state["query"]
    context_docs = state["context"]
    context_parts = []
    for doc in context_docs:
      title = doc.metadata.get("issue_title", "Unknown")
      context_parts.append(f"[Issue: {title}]\n{doc.page_content}")

    context_str = "\n\n---\n\n".join(context_parts)
    prompt = support_prompt.invoke({"context": context_str, "query": query})
    response = support_llm.invoke(prompt)
    return {
        "query": query,
        "context": context_docs,
        "answer": response.content
    }

## Part 4: Build the Workflow

Create the LangGraph workflow that connects the retrieve and generate nodes.

In [ ]:
# Build the RAG workflow

# TODO 1: Create a StateGraph with SupportRAGState
support_workflow = StateGraph(SupportRAGState)  # Replace with StateGraph(SupportRAGState)

# TODO 2: Add the retrieve node (name it "retrieve")
support_workflow.add_node("retrieve", retrieve_support_docs)

# TODO 3: Add the generate node (name it "generate")
support_workflow.add_node("generate", generate_support_response)

# TODO 4: Add edge from START to "retrieve"
support_workflow.add_edge(START, "retrieve")

# TODO 5: Add edge from "retrieve" to "generate"
support_workflow.add_edge("retrieve", "generate")

# TODO 6: Add edge from "generate" to END
support_workflow.add_edge("generate", END)

# Compile the workflow
support_rag = support_workflow.compile()

print("Workflow compiled!")
print("\nWorkflow structure: START -> retrieve -> generate -> END")

Workflow compiled!

Workflow structure: START -> retrieve -> generate -> END


## Run Your Implementation

Test your RAG pipeline with sample customer questions.

In [ ]:
# Helper function to query the support system

def ask_support(question: str):
    """Query the TechMart support RAG system."""
    print("=" * 70)
    print(f"Customer Question: {question}")
    print("=" * 70)

    result = support_rag.invoke({
        "query": question,
        "context": [],
        "answer": ""
    })

    print("\nSources consulted:")
    for doc in result["context"]:
        print(f"  - {doc.metadata.get('issue_title', 'Unknown')} ({doc.metadata.get('product_id', 'Unknown')})")

    print(f"\nSupport Response:\n{result['answer']}")
    print("\n")

In [ ]:
# Test 1: Battery issue
ask_support("My UltraBook Pro battery is draining really fast, what should I do?")

Customer Question: My UltraBook Pro battery is draining really fast, what should I do?

Sources consulted:
  - UltraBook Pro battery drains quickly (SKU001)
  - UltraBook Pro overheating under load (SKU001)
  - UltraBook Air fan noise while idle (SKU002)
  - UltraBook Air Wi‑Fi disconnects frequently (SKU002)

Support Response:
Here are some troubleshooting steps you can follow to address the battery drain issue on your UltraBook Pro:

1. Switch Windows power mode to ‘Balanced’.
2. Disable the OLED panel’s HDR in Display Settings.
3. Check Task Manager for apps with more than 10% CPU usage and close them.
4. Update BIOS to version 1.09, which optimizes idle power draw.

Try these steps and see if they help improve your battery life. If the issue persists, please contact support for further assistance.




In [ ]:
# Test 2: Connectivity issue
ask_support("My wireless mouse keeps disconnecting every few minutes")

Customer Question: My wireless mouse keeps disconnecting every few minutes

Sources consulted:
  - Wireless mouse disconnects randomly (SKU010)
  - Mouse Pointer Lagging or Stuttering During Use (SKU236)
  - Mouse Pointer Lag or Stuttering in Wireless Mode (SKU035)
  - UltraBook Air Wi‑Fi disconnects frequently (SKU002)

Support Response:
Here are some troubleshooting steps you can follow to resolve the issue with your wireless mouse disconnecting randomly:

1. Move the receiver to a front USB port to reduce interference.
2. Replace the AA battery or charge the mouse for at least 30 minutes.
3. Toggle the 2.4 GHz / Bluetooth switch and re-pair the mouse.
4. Update the firmware via GlideMouse Utility and reboot your PC.

Try these steps and see if the issue persists. If it does, please let me know!




In [ ]:
# Test 3: Display issue
ask_support("The monitor won't turn on after my computer wakes from sleep")

Customer Question: The monitor won't turn on after my computer wakes from sleep

Sources consulted:
  - VisionView 27 monitor won’t turn on (SKU005)
  - No Signal Detected on HDMI 2.1 Port (SKU205)
  - No Signal Detected on DisplayPort Connection (SKU240)
  - Desktop Powers On But No Display Output (SKU068)

Support Response:
It sounds like you're experiencing an issue with your VisionView 27 monitor not turning on after your computer wakes from sleep. Here are some troubleshooting steps you can follow:

1. Verify that the power brick is firmly clicked into the monitor port.
2. Test a different outlet to ensure the power source is working.
3. Hold the monitor power button for 10 seconds to perform a hard reset.
4. Swap the power cable; if the LED stays off, the power brick may be faulty.

If the monitor still does not turn on after trying these steps, please let me know, and I can assist you further!


